<a href="https://colab.research.google.com/github/adelinewidyatmoko/ProjectA_Kelompok6_BanjirArticles_PBA/blob/main/notebooks/01_scraping_berita_banjir_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraping Artikel Berita Banjir


In [ ]:
!pip install newspaper3k lxml_html_clean beautifulsoup4 -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 60.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import time
import random
import re
import os
import requests
from bs4 import BeautifulSoup
from newspaper import Article

## 1. Load & Bersihkan Data

In [ ]:
df = pd.read_csv('data_all.csv')
df.columns = df.columns.str.strip()
print(f'Total data: {len(df)}')
print(f'Kolom: {list(df.columns)}')
print(f'\nData kosong per kolom:')
print(df.isnull().sum())
df.head()

Total data: 2492
Kolom: ['Judul', 'Link', 'Sumber', 'Sentimen']

Data kosong per kolom:
Judul       1197
Link           1
Sumber      1598
Sentimen       0
dtype: int64


,Judul,Link,Sumber,Sentimen
0,Area Terbangun di DAS Badung dan Mati Memperpa...,https://www.kompas.id/artikel/kawasan-terbangu...,Kompas,Banjir
1,"Banjir Besar di Bali, Budayawan Sebut Larangan...",https://denpasar.kompas.com/read/2025/09/11/09...,Kompas,Banjir
2,Turis Asing Pakai Perahu Karet Lintasi Genanga...,https://denpasar.kompas.com/read/2025/09/10/15...,Kompas,Banjir
3,"Banjir di Jalan Paccerakkang Makassar, Ruko Te...",https://www.detik.com/sulsel/makassar/d-837192...,Detik,Banjir
4,"Kementerian LH Bakal Evaluasi Tata Ruang Bali,...",https://denpasar.kompas.com/read/2025/09/15/13...,Kompas,Banjir


In [ ]:
# hapus baris tanpa link
df = df.dropna(subset=['Link'])
df = df[df['Link'].str.strip() != '']

# cek duplikat
dup_count = df.duplicated(subset=['Link'], keep=False).sum()
print(f'Baris duplikat: {dup_count}')

# hapus duplikat, keep yang paling lengkap (paling sedikit kolom kosong)
df['null_count'] = df.isnull().sum(axis=1)
df = df.sort_values('null_count').drop_duplicates(subset=['Link'], keep='first')
df = df.drop(columns='null_count')
df = df.reset_index(drop=True)

print(f'Data setelah bersih: {len(df)}')

Baris duplikat: 74
Data setelah bersih: 2454


## 2. Deteksi Sumber dari URL

In [ ]:
def detect_sumber(url):
    url = str(url).lower()
    if 'kompas' in url: return 'Kompas'
    if 'tribunnews' in url or 'tribun-' in url: return 'Tribun News'
    if 'detik' in url: return 'Detik'
    if 'antaranews' in url: return 'Antara News'
    if 'cnnindonesia' in url: return 'CNN Indonesia'
    if 'liputan6' in url: return 'Liputan6'
    if 'tempo' in url: return 'Tempo'
    if 'bbc' in url: return 'BBC'
    if 'inews' in url: return 'iNews'
    if 'jawapos' in url or 'radar' in url: return 'Jawa Pos'
    if 'suara' in url: return 'Suara.com'
    if 'okezone' in url: return 'Okezone'
    if 'merdeka' in url: return 'Merdeka'
    if 'sindonews' in url: return 'SINDOnews'
    if 'republika' in url: return 'Republika'
    if 'idntimes' in url: return 'IDN Times'
    if 'viva' in url: return 'Viva'
    if 'kaltimpost' in url: return 'Kaltim Post'
    if 'klikkalsel' in url: return 'KlikKalsel'
    return 'Lainnya'

# isi sumber yang kosong, update yang tadinya google news
df['Sumber'] = df.apply(
    lambda row: row['Sumber'] if pd.notna(row['Sumber']) and str(row['Sumber']).strip() not in ['', 'nan']
    else detect_sumber(row['Link']), axis=1
)

print('Distribusi sumber:')
print(df['Sumber'].value_counts())

Distribusi sumber:
Sumber
Detik              624
Lainnya            441
Tribun News        385
CNN Indonesia      154
Liputan6           121
                  ... 
Netral News          1
Berita Nasional      1
Kase Data            1
IDN Times            1
Berita Satu          1
Name: count, Length: 234, dtype: int64


## 3. Scraping Isi Artikel
Pakai newspaper3k sebagai cara utama. Kalau gagal, fallback ke requests + BeautifulSoup.

Progress disave otomatis tiap 100 artikel, jadi kalau Colab disconnect tinggal jalankan ulang dari cell checkpoint.

In [ ]:
def scrape_newspaper(url):
    """cara utama: pakai newspaper3k"""
    art = Article(url, language='id')
    art.download()
    art.parse()
    if art.text and len(art.text.strip()) > 50:
        return {
            'judul': art.title or '',
            'konten': art.text,
            'tanggal': str(art.publish_date) if art.publish_date else ''
        }
    return None

def scrape_bs4(url):
    """fallback: pakai requests + beautifulsoup"""
    r = requests.get(url, headers=HEADERS, timeout=15)
    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, 'html.parser')

    # hapus script, style, nav, footer
    for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
        tag.decompose()

    # coba ambil judul
    judul = ''
    title_tag = soup.find('h1')
    if title_tag:
        judul = title_tag.get_text(strip=True)

    # coba ambil konten dari tag article atau div content
    konten = ''
    selectors = ['article', 'div.content-body', 'div.detail__body-text',
                 'div.read__content', 'div.post-content', 'div.entry-content',
                 'div.artikel', 'div.detail-text', 'div.text-article']

    for sel in selectors:
        el = soup.select_one(sel)
        if el:
            paragraphs = el.find_all('p')
            if paragraphs:
                konten = ' '.join(p.get_text(strip=True) for p in paragraphs)
                break

    # kalo masih kosong, ambil semua <p> di body
    if not konten or len(konten) < 50:
        body = soup.find('body')
        if body:
            paragraphs = body.find_all('p')
            texts = [p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 30]
            konten = ' '.join(texts)

    if konten and len(konten.strip()) > 50:
        return {'judul': judul, 'konten': konten, 'tanggal': ''}
    return None

def scrape_artikel(url, max_retries=2):
    """coba scrape dengan newspaper dulu, fallback ke bs4"""
    for attempt in range(max_retries):
        try:
            # coba newspaper3k dulu
            result = scrape_newspaper(url)
            if result:
                return result
        except:
            pass

        try:
            # fallback ke bs4
            result = scrape_bs4(url)
            if result:
                return result
        except:
            pass

        # tunggu sebentar sebelum retry
        if attempt < max_retries - 1:
            time.sleep(2)

    return None

In [ ]:
# siapkan kolom baru
checkpoint = '/content/scraping_checkpoint.csv'
start_idx = 0

if os.path.exists(checkpoint):
    df = pd.read_csv(checkpoint)
    # cari index terakhir yang udah ada konten
    has_konten = df['Konten'].notna() & (df['Konten'].astype(str).str.strip() != '')
    if has_konten.any():
        start_idx = has_konten[::-1].idxmax() + 1
    print(f'Checkpoint ditemukan, lanjut dari index {start_idx}')
else:
    df['Konten'] = ''
    df['Tanggal'] = ''
    print('Mulai dari awal')

print(f'Total yang perlu di-scrape: {len(df) - start_idx}')

Mulai dari awal
Total yang perlu di-scrape: 2454


In [ ]:
# mulai scraping
total = len(df)
berhasil = 0
gagal = []

for i in range(start_idx, total):
    url = str(df.at[i, 'Link']).strip()

    # skip kalo url ga valid
    if not url.startswith('http'):
        gagal.append(i)
        continue

    print(f'[{i+1}/{total}] {url[:75]}...', end=' ')

    result = scrape_artikel(url)

    if result:
        df.at[i, 'Konten'] = result['konten']
        if result['tanggal']:
            df.at[i, 'Tanggal'] = result['tanggal']
        # isi judul kalo kosong
        if pd.isna(df.at[i, 'Judul']) or str(df.at[i, 'Judul']).strip() == '':
            if result['judul']:
                df.at[i, 'Judul'] = result['judul']
        berhasil += 1
        print('OK')
    else:
        gagal.append(i)
        print('GAGAL')

    # checkpoint tiap 100
    if (i + 1) % 100 == 0:
        df.to_csv(checkpoint, index=False)
        pct = berhasil / (i - start_idx + 1) * 100 if i > start_idx else 0
        print(f'\n--- Checkpoint ({i+1}/{total}) | Success rate: {pct:.0f}% ---\n')

    time.sleep(random.uniform(1.5, 3.0))

# final save
df.to_csv(checkpoint, index=False)
print(f'\n{"="*40}')
print(f'Selesai! Berhasil: {berhasil}, Gagal: {len(gagal)}')

[1/2454] https://www.detik.com/sulsel/makassar/d-8371822/akses-jalan-perumnas-antang... OK
[2/2454] https://news.detik.com/berita/d-8371585/banjir-di-makassar-545-warga-pengun... OK
[3/2454] https://www.detik.com/sulsel/makassar/d-8371546/motor-mogok-di-jalan-inspek... OK
[4/2454] https://bali.tribunnews.com/denpasar/592348/3-smp-dan-beberapa-sd-terendam-... GAGAL
[5/2454] https://bali.tribunnews.com/buleleng/589486/genangan-air-capai-1-m-puluhan-... GAGAL
[6/2454] https://bali.tribunnews.com/2021/10/06/antisipasi-banjir-saat-musim-hujan-p... GAGAL
[7/2454] https://bali.tribunnews.com/2021/11/20/beredar-video-pintu-masuk-pura-uluwa... GAGAL
[8/2454] https://bali.tribunnews.com/2015/03/12/empat-desa-di-buleleng-terendam-banj... GAGAL
[9/2454] https://bali.tribunnews.com/2021/12/06/akibat-hujan-deras-guyur-denpasar-se... GAGAL
[10/2454] https://bali.tribunnews.com/denpasar/592337/33-titik-di-denpasar-bali-teren... GAGAL
[11/2454] https://bali.tribunnews.com/bali/587789/biang-kerok-banjir

## 4. Isi Judul yang Kosong

In [ ]:
# cek judul kosong
judul_kosong = df['Judul'].isna() | (df['Judul'].astype(str).str.strip() == '')
print(f'Judul masih kosong: {judul_kosong.sum()}')

# kalau masih ada yg kosong, coba ambil dari tag <title> di link
if judul_kosong.sum() > 0:
    for idx in df[judul_kosong].index:
        url = str(df.at[idx, 'Link'])
        if not url.startswith('http'):
            continue
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(r.text, 'html.parser')
            h1 = soup.find('h1')
            if h1:
                df.at[idx, 'Judul'] = h1.get_text(strip=True)
            elif soup.title:
                df.at[idx, 'Judul'] = soup.title.get_text(strip=True)
            time.sleep(1)
        except:
            pass

    judul_masih = df['Judul'].isna() | (df['Judul'].astype(str).str.strip() == '')
    print(f'Judul masih kosong setelah retry: {judul_masih.sum()}')

Judul masih kosong: 205
Judul masih kosong setelah retry: 205


## 5. Lihat yang Gagal

In [ ]:
if gagal:
    print(f'Total gagal: {len(gagal)}\n')
    print('20 link pertama yang gagal:')
    for idx in gagal[:20]:
        print(f'  [{idx}] {df.at[idx, "Link"]}')

Total gagal: 800

20 link pertama yang gagal:
  [3] https://bali.tribunnews.com/denpasar/592348/3-smp-dan-beberapa-sd-terendam-banjir-di-denpasar-siswa-belajar-daring
  [4] https://bali.tribunnews.com/buleleng/589486/genangan-air-capai-1-m-puluhan-rumah-di-pancasari-terendambanjir-landa-lima-banjar-di-pancasari
  [5] https://bali.tribunnews.com/2021/10/06/antisipasi-banjir-saat-musim-hujan-pupr-kota-denpasar-gelar-pembersihan-drainase-di-beberapa-titik
  [6] https://bali.tribunnews.com/2021/11/20/beredar-video-pintu-masuk-pura-uluwatu-terendam-banjir-bpbd-badung-tidak-ada-laporan-resmi-masuk
  [7] https://bali.tribunnews.com/2015/03/12/empat-desa-di-buleleng-terendam-banjir
  [8] https://bali.tribunnews.com/2021/12/06/akibat-hujan-deras-guyur-denpasar-selain-rumah-warga-rsup-sanglah-juga-tergenang-air
  [9] https://bali.tribunnews.com/denpasar/592337/33-titik-di-denpasar-bali-terendam-banjir-tanggul-sungai-jebol-di-siulan
  [10] https://bali.tribunnews.com/bali/587789/biang-kerok-banji

## 6. Simpan Hasil Final

In [ ]:
# susun kolom
cols = ['Judul', 'Link', 'Sumber', 'Sentimen', 'Konten', 'Tanggal']
for c in cols:
    if c not in df.columns:
        df[c] = ''

df_final = df[cols].copy()
df_final = df_final.reset_index(drop=True)
df_final.index += 1
df_final.index.name = 'No'

konten_ok = df_final['Konten'].notna() & (df_final['Konten'].astype(str).str.strip() != '')

print('=== RINGKASAN ===')
print(f'Total berita: {len(df_final)}')
print(f'Konten terisi: {konten_ok.sum()} ({konten_ok.sum()/len(df_final)*100:.1f}%)')
print(f'Konten kosong: {(~konten_ok).sum()}')
print(f'Judul kosong: {df_final["Judul"].isna().sum()}')
print(f'\nDistribusi sumber:')
print(df_final['Sumber'].value_counts())
print(f'\nDistribusi sentimen:')
print(df_final['Sentimen'].value_counts())

=== RINGKASAN ===
Total berita: 2454
Konten terisi: 1654 (67.4%)
Konten kosong: 800
Judul kosong: 205

Distribusi sumber:
Sumber
Detik              624
Lainnya            441
Tribun News        385
CNN Indonesia      154
Liputan6           121
                  ... 
Netral News          1
Berita Nasional      1
Kase Data            1
IDN Times            1
Berita Satu          1
Name: count, Length: 234, dtype: int64

Distribusi sentimen:
Sentimen
Banjir     1278
Bandang    1176
Name: count, dtype: int64


In [ ]:
output = 'data_berita_banjir_scraped.csv'
df_final.to_csv(output)
print(f'Tersimpan: {output}')

from google.colab import files
files.download(output)

Tersimpan: data_berita_banjir_scraped.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>